# Demand Forecasting
**AeroNet Lite - ML Pipeline (Regression)**

Trains LinearRegression and RandomForestRegressor on **real Kaggle Bike Sharing Demand data**.
Reports MAE and RMSE for both models.

## Download Bike Sharing Data (No Authentication Needed)
Direct download from public source - no API key required!

In [ ]:
import os
import requests

data_dir = "bike_sharing_data"
os.makedirs(data_dir, exist_ok=True)

print("Downloading Kaggle Bike Sharing Demand dataset...\n")

# Download from public GitHub mirror
files = {
    'train.csv': 'https://raw.githubusercontent.com/dataoptimal/datasets/master/Kaggle_Bike_Sharing_Demand/train.csv',
    'test.csv': 'https://raw.githubusercontent.com/dataoptimal/datasets/master/Kaggle_Bike_Sharing_Demand/test.csv',
}

for filename, url in files.items():
    try:
        print(f"Downloading {filename}...")
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        filepath = f'{data_dir}/{filename}'
        with open(filepath, 'wb') as f:
            f.write(response.content)
        
        size_kb = len(response.content) / 1024
        print(f"✓ {filename} ({size_kb:.1f} KB)\n")
    except Exception as e:
        print(f"✗ Failed to download {filename}: {e}\n")

print(f"✓ Data ready in '{data_dir}/' directory\n")
print("Files downloaded:")
for file in sorted(os.listdir(data_dir)):
    file_path = os.path.join(data_dir, file)
    if os.path.isfile(file_path):
        file_size = os.path.getsize(file_path) / 1024
        print(f"  ✓ {file} ({file_size:.1f} KB)")

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore Real Data

In [ ]:
# Load train and test data from Kaggle
train_df = pd.read_csv('bike_sharing_data/train.csv')
test_df = pd.read_csv('bike_sharing_data/test.csv')

print(f"Train dataset shape: {train_df.shape}")
print(f"Test dataset shape:  {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"\nFirst few rows of train data:")
print(train_df.head(10))

In [ ]:
# Data info
print("Train Data Info:")
print(train_df.info())
print("\nMissing values:")
print(train_df.isnull().sum())
print("\nBasic Statistics:")
print(train_df.describe())

## 3. Feature Engineering

In [ ]:
# Create working copy
df = train_df.copy()

# Convert datetime
df['datetime'] = pd.to_datetime(df['datetime'])

# Extract time-based features
df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.day
df['month'] = df['datetime'].dt.month
df['dayofweek'] = df['datetime'].dt.dayofweek
df['dayofyear'] = df['datetime'].dt.dayofyear

# Drop datetime column as we've extracted features
df = df.drop(['datetime'], axis=1)

print("Features after engineering:")
print(df.columns.tolist())
print(f"\nDataset shape: {df.shape}")
print(df.head())

## 4. Exploratory Data Analysis

In [ ]:
# Basic stats
print("Descriptive Statistics:")
print(df.describe())
print("\nCorrelation with Count (Demand):")
corr_with_count = df.corr()['count'].sort_values(ascending=False)
print(corr_with_count)

In [ ]:
# Demand distribution and trends
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of demand
axes[0, 0].hist(df['count'], bins=50, color='#2196F3', edgecolor='white', alpha=0.7)
axes[0, 0].set_title('Bike Demand Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Count (Bikes Rented)')
axes[0, 0].set_ylabel('Frequency')

# Demand by hour
hourly_avg = df.groupby('hour')['count'].mean()
axes[0, 1].plot(hourly_avg.index, hourly_avg.values, marker='o', color='#4CAF50', linewidth=2)
axes[0, 1].set_title('Average Demand by Hour', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Average Count')
axes[0, 1].grid(alpha=0.3)

# Demand by day of week
daily_avg = df.groupby('dayofweek')['count'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 0].bar(range(7), daily_avg.values, color='#FF9800', alpha=0.7, edgecolor='black')
axes[1, 0].set_xticks(range(7))
axes[1, 0].set_xticklabels(days)
axes[1, 0].set_title('Average Demand by Day of Week', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Average Count')

# Temperature vs Demand
axes[1, 1].scatter(df['temp'], df['count'], alpha=0.3, c='#9C27B0', s=10)
axes[1, 1].set_title('Temperature vs Demand', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Temperature (°C)')
axes[1, 1].set_ylabel('Count (Bikes Rented)')

plt.tight_layout()
plt.show()

## 5. Prepare Data for Modeling

In [ ]:
# Select features for modeling
# Using both casual and registered will leak information, so we drop them
feature_cols = ['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 
                 'humidity', 'windspeed', 'hour', 'day', 'month', 'dayofweek']

X = df[feature_cols].copy()
y = df['count'].copy()  # Target: total bike rentals

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature list: {feature_cols}")
print(f"\nTarget (count) statistics:")
print(y.describe())

In [ ]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"Training target mean: {y_train.mean():.2f}")
print(f"Test target mean: {y_test.mean():.2f}")

## 6. Train Models

In [ ]:
# Train Linear Regression
print("Training LinearRegression...")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test)

# Evaluate
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = lr_model.score(X_test, y_test)

print(f"\nLinearRegression Performance:")
print(f"  MAE  = {mae_lr:.3f}")
print(f"  RMSE = {rmse_lr:.3f}")
print(f"  R²   = {r2_lr:.3f}")

In [ ]:
# Train Random Forest Regressor
print("Training RandomForestRegressor...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, 
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = rf_model.score(X_test, y_test)

print(f"\nRandomForestRegressor Performance:")
print(f"  MAE  = {mae_rf:.3f}")
print(f"  RMSE = {rmse_rf:.3f}")
print(f"  R²   = {r2_rf:.3f}")

## 7. Model Comparison & Visualization

In [ ]:
# Create comparison bar chart
models = ['LinearRegression', 'RandomForest']
mae_values = [mae_lr, mae_rf]
rmse_values = [rmse_lr, rmse_rf]
r2_values = [r2_lr, r2_rf]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# MAE comparison
axes[0].bar(models, mae_values, color=['#2196F3', '#4CAF50'], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Mean Absolute Error')
axes[0].set_title('MAE Comparison', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(mae_values):
    axes[0].text(i, v + 5, f'{v:.1f}', ha='center', fontweight='bold')

# RMSE comparison
axes[1].bar(models, rmse_values, color=['#FF9800', '#F44336'], alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Root Mean Squared Error')
axes[1].set_title('RMSE Comparison', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(rmse_values):
    axes[1].text(i, v + 10, f'{v:.1f}', ha='center', fontweight='bold')

# R² comparison
axes[2].bar(models, r2_values, color=['#9C27B0', '#00BCD4'], alpha=0.7, edgecolor='black')
axes[2].set_ylabel('R² Score')
axes[2].set_title('R² Comparison', fontweight='bold')
axes[2].set_ylim([0, 1])
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(r2_values):
    axes[2].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Determine best model
best_model = "RandomForest" if r2_rf > r2_lr else "LinearRegression"
print(f"\n✓ Best Model: {best_model}")

In [ ]:
# Actual vs Predicted scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression
axes[0].scatter(y_test, y_pred_lr, alpha=0.5, c='#2196F3', s=20, edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
              'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_title('LinearRegression: Actual vs Predicted', fontweight='bold')
axes[0].set_xlabel('Actual Count')
axes[0].set_ylabel('Predicted Count')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Random Forest
axes[1].scatter(y_test, y_pred_rf, alpha=0.5, c='#4CAF50', s=20, edgecolors='black', linewidth=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
              'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_title('RandomForest: Actual vs Predicted', fontweight='bold')
axes[1].set_xlabel('Actual Count')
axes[1].set_ylabel('Predicted Count')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Random Forest feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_importance['feature'], feature_importance['importance'], 
        color='#9C27B0', alpha=0.7, edgecolor='black')
ax.set_xlabel('Importance', fontweight='bold')
ax.set_title('RandomForest Feature Importance', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nFeature Importance Ranking:")
print(feature_importance.sort_values('importance', ascending=False).to_string(index=False))

## 9. Residual Analysis

In [ ]:
# Calculate residuals
residuals_lr = y_test - y_pred_lr
residuals_rf = y_test - y_pred_rf

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# LR residuals histogram
axes[0, 0].hist(residuals_lr, bins=40, color='#2196F3', edgecolor='white', alpha=0.7)
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title('LinearRegression: Residuals Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Residual')
axes[0, 0].set_ylabel('Frequency')

# RF residuals histogram
axes[0, 1].hist(residuals_rf, bins=40, color='#4CAF50', edgecolor='white', alpha=0.7)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_title('RandomForest: Residuals Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Residual')
axes[0, 1].set_ylabel('Frequency')

# LR residuals scatter
axes[1, 0].scatter(y_pred_lr, residuals_lr, alpha=0.5, c='#2196F3', s=20)
axes[1, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_title('LinearRegression: Residuals vs Predicted', fontweight='bold')
axes[1, 0].set_xlabel('Predicted Count')
axes[1, 0].set_ylabel('Residual')
axes[1, 0].grid(alpha=0.3)

# RF residuals scatter
axes[1, 1].scatter(y_pred_rf, residuals_rf, alpha=0.5, c='#4CAF50', s=20)
axes[1, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_title('RandomForest: Residuals vs Predicted', fontweight='bold')
axes[1, 1].set_xlabel('Predicted Count')
axes[1, 1].set_ylabel('Residual')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion & Summary

This notebook demonstrates demand forecasting using **real Kaggle Bike Sharing data** instead of synthetic data.

### Key Findings:
- **RandomForest** typically outperforms LinearRegression on this dataset (handles non-linearity)
- **Most important features**: hour, temperature, season, and workingday
- **Demand patterns**: Clear peaks during commuting hours (7-9 AM, 4-6 PM)
- **Seasonal trends**: Weather and temperature significantly impact bike rentals

### Next Steps:
1. Experiment with more advanced models (XGBoost, LightGBM, Neural Networks)
2. Fine-tune hyperparameters using cross-validation
3. Apply time-series specific techniques (ARIMA, Prophet)
4. Use predictions for fleet management and delivery optimization